# C-04 Bias Detection in Data

**Control:** C-04 Bias Detection in Data, MAK-AI Responsible AI Toolkit.

**Learning goal:** detect three kinds of bias before any model is trained: historical bias (different label rates across groups), proxy variables (features that stand in for a protected attribute) and labelling bias (annotators who disagree), and decide what to do about each.

**Time to run:** about 2 minutes on an ordinary laptop. No GPU.

**Licence:** this notebook is Apache-2.0.

**Example data:**
- UCI Adult (Becker and Kohavi, 1996), DOI 10.24432/C5XW20, licence CC BY 4.0, 48,842 rows, from https://archive.ics.uci.edu/dataset/2/adult.
- A small **synthetic** table of three annotators' labels, generated inside the notebook from a fixed seed, because Adult has one label per row. It is invented, labelled as synthetic wherever it appears, and says nothing about real annotators.

Adult is credited here because CC BY 4.0 asks for attribution.

**A caveat before you start:** Adult is a 1994 United States census extract, with that year's categories for sex and race. The notebook teaches a method; its numbers say nothing about the people your system serves. To choose the groups that matter where you work, ask who your system's decisions fall on, which groups have been treated unequally in your setting (for example by language, region, disability or displacement), and who knows how those groups are recorded in your data. Name those columns in `GROUP_COLUMNS` and `PROTECTED` in the "Adapt here" cell. A group that is not a column cannot be checked at all (Step 5).

## Linked risk areas and questions

Generated from content 5.2.0. Do not edit by hand.

Risk areas: 3

- PP-02: Data Scarcity & Availability Bias
- PP-03: Historical & Societal Bias
- PP-07: Labeling Bias

Questions: 13: Q-PP-04, Q-PP-05, Q-PP-24, Q-PP-09, Q-PP-11, Q-PP-12, Q-PP-16, Q-PP-17, Q-PP-18, Q-PP-19, Q-PP-20, Q-PP-22, Q-PP-23

## When to use this notebook

Use it when you are answering questions about bias in the data and your status is **Not yet** or **Partly**, especially with the reason code `ny_needs_skills` ("Needs skills or guidance the team does not have yet"). It helps with:

- `Q-PP-11`: have you identified and addressed proxy variables for protected attributes?
- `Q-PP-12`: have you ensured the quality and consistency of data labels and annotations?
- `Q-PP-17`: which bias types have you assessed for in your data?
- `Q-PP-19`: have you evaluated whether historical patterns in the data reflect past discrimination or systemic inequities?

You need a table with one row per person, the outcome label, the columns you plan to use, and the columns that define the groups you worry about. For the labelling part you need the labels from at least two annotators for the same items.

In [ ]:
import hashlib
import importlib.metadata
import io
import os
import platform
import tempfile
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa
from statsmodels.stats.proportion import proportion_confint

SEED = 20261030  # the one random seed; every random step below uses it
rng = np.random.default_rng(SEED)
pd.set_option('display.width', 150)
pd.set_option('display.max_columns', 30)

PACKAGES = ['pandas', 'numpy', 'scipy', 'scikit-learn', 'statsmodels', 'matplotlib']

print('Python', platform.python_version())
for _package in PACKAGES:
    print(_package, importlib.metadata.version(_package))

In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/2/data.csv'
DATA_SHA256 = '26b38f8623766396e2ba0d1a235a283db8ed993cc32bf32fc8aab09708c3bdf2'
DATA_FILE = 'uci-adult-2-data.csv'  # the cache file name is fixed here

CACHE_DIR = Path.home() / '.cache' / 'makrai-notebooks'


MAX_DOWNLOAD_BYTES = 64 * 1024 * 1024  # a dataset file is never larger than this; a longer body is refused


def fetch_checked(url, sha256, file_name):
    """Return the bytes of a dataset whose SHA-256 matches `sha256`.

    The file is downloaded once into the cache folder, and written there only after its hash
    matches. Every run, including the first, hashes the bytes before they are used, and a
    mismatch stops the notebook. The body is read in 1 MiB pieces and refused past MAX_DOWNLOAD_BYTES.
    `file_name` is fixed in this notebook, never taken from the address.
    """
    if not url.startswith('https://'):
        raise ValueError(f'the dataset address must be https: {url}')
    if Path(file_name).name != file_name:
        raise ValueError(f'the cache file name must be a plain name: {file_name}')

    def read_checked(stream):
        digest, pieces, size = hashlib.sha256(), [], 0
        for piece in iter(lambda: stream.read(1 << 20), b''):
            size += len(piece)
            if size > MAX_DOWNLOAD_BYTES:
                raise RuntimeError(f'{file_name}: the file is larger than {MAX_DOWNLOAD_BYTES:,} bytes; refused')
            digest.update(piece)
            pieces.append(piece)
        return b''.join(pieces), digest.hexdigest()

    path = CACHE_DIR / file_name
    if path.exists():
        with path.open('rb') as stream:
            data, found = read_checked(stream)
        source, remedy = f'the cached file {path}', f'Delete {path} and run again.'
    else:
        request = urllib.request.Request(url, headers={'User-Agent': 'makrai-notebooks'})
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                if not response.geturl().startswith('https://'):
                    raise RuntimeError(f'{url} redirected to a non-https address: {response.geturl()}')
                data, found = read_checked(response)
        except OSError as error:
            raise RuntimeError(f'could not download {url}: {error}. If you cannot reach it, save the file by hand '
                               f'as {path}; it is checked against the pinned SHA-256 either way.') from error
        source, remedy = url, 'Do not use this download; check the dataset page for a new version.'
    if found != sha256:
        raise RuntimeError(f'{file_name}: the SHA-256 of {source} is {found}, expected {sha256}. {remedy}')
    if not path.exists():
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        partial = tempfile.NamedTemporaryFile(dir=CACHE_DIR, suffix='.part', delete=False)
        try:
            with partial:
                partial.write(data)
            os.replace(partial.name, path)  # the cache never holds a half-written file, whoever else is writing
        except BaseException:
            Path(partial.name).unlink(missing_ok=True)
            raise
    return data


# To use your own file, replace the next line with:  raw = Path('your-file.csv').read_bytes()
raw = fetch_checked(DATA_URL, DATA_SHA256, DATA_FILE)
DATA_SHA256_USED = hashlib.sha256(raw).hexdigest()  # the evidence summary records the hash of the bytes used
df = pd.read_csv(io.BytesIO(raw))
df['income'] = df['income'].str.rstrip('.')  # Adult only: the census test file writes the label with a trailing dot
print(f'{len(df):,} rows, {df.shape[1]} columns; SHA-256 of the data used: {DATA_SHA256_USED[:12]}...')

# SYNTHETIC: three annotators label 900 invented items, in two groups of items. Annotator 3 is made to label group B
# items as negative more often than the others do. This table is made up from the seed above and describes no real people.
SYNTHETIC = True
synthetic_rng = np.random.default_rng(SEED + 1)
n_items = 900
item_group = np.where(synthetic_rng.random(n_items) < 0.5, 'group A', 'group B')
truth = synthetic_rng.random(n_items) < 0.4
annotations = pd.DataFrame({'item_group': item_group})
for number, accuracy in enumerate([0.92, 0.90, 0.88], start=1):
    correct = synthetic_rng.random(n_items) < accuracy
    label = np.where(correct, truth, ~truth)
    if number == 3:  # the planted labelling bias
        label = np.where((item_group == 'group B') & truth & (synthetic_rng.random(n_items) < 0.35), False, label)
    annotations[f'annotator_{number}'] = label.astype(int)
print(f'synthetic annotation table: {len(annotations)} items, 3 annotators (made up, seed {SEED + 1})')

In [ ]:
# ADAPT HERE. Edit only this cell to run the notebook on your own data. Every later cell reads only these names.
LABEL = 'income'                    # the column that holds the outcome label
FAVOURABLE = '>50K'                 # the value of LABEL that counts as the favourable outcome
PROTECTED = 'sex'                   # the one column for the proxy test: can the other columns predict it?
GROUP_COLUMNS = ['sex', 'race']     # columns that define the groups whose label rates you compare
NUMERIC = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
CATEGORICAL = ['workclass', 'marital-status', 'occupation', 'relationship', 'native-country']
ANNOTATOR_COLUMNS = ['annotator_1', 'annotator_2', 'annotator_3']   # one column of labels (0 or 1) per annotator
ANNOTATION_GROUP = 'item_group'     # the column that says which group each annotated item belongs to
MIN_GROUP_SIZE = 100                # groups smaller than this are flagged, not interpreted
PROXY_MARGIN = 0.10                 # accuracy this far above guessing the common value counts as a proxy signal

### Step 1: historical bias, label rates by group and by intersection

**What this does.** It reports, for each group and for each combination of the group columns, the share with the favourable label, with a 95 percent interval (the Wilson interval from statsmodels), and a chi-square test of whether label and group are independent. It also reports Cramer's V, an effect size between 0 (no association) and 1 (complete), because with tens of thousands of rows the test's p-value is tiny even for a small difference.

**What it shows.** Whether the label was handed out at different rates, which is what a model trained on it will learn.

**What it does not show.** Whether the gap is unfair. A gap can reflect real differences in the world, differences in how people were treated, or how the label was made. That is the "Stop and decide" question.

In [ ]:
y = (df[LABEL] == FAVOURABLE).astype(int)
groups = df[GROUP_COLUMNS].fillna('missing').astype(str)
groupings = {name: groups[name] for name in GROUP_COLUMNS}
groupings['intersection'] = groups.agg(' / '.join, axis=1)

rate_rows, test_rows = [], []
for name, feature in groupings.items():
    counts = y.groupby(feature).agg(['sum', 'size'])
    low, high = (np.asarray(v) for v in proportion_confint(counts['sum'], counts['size'], method='wilson'))
    for position, (label, row) in enumerate(counts.iterrows()):
        rate_rows.append({'grouping': name, 'group': label, 'people': int(row['size']), 'favourable share': row['sum'] / row['size'],
                          'low': low[position], 'high': high[position], 'small': row['size'] < MIN_GROUP_SIZE})
    table = pd.crosstab(feature, y)
    chi2, p_value, _, _ = chi2_contingency(table)
    test_rows.append({'grouping': name, 'chi-square': chi2, 'p-value': p_value, 'Cramers V': association(table.to_numpy(), method='cramer')})

label_rates = pd.DataFrame(rate_rows)
label_tests = pd.DataFrame(test_rows)
print(label_rates.round(3).to_string(index=False))
print()
print(label_tests.round(4).to_string(index=False))

### Step 2: proxy variables, how strongly each column is tied to the protected attribute

**What this does.** For each column it measures how strongly it is associated with `PROTECTED`. For categorical columns that is Cramer's V from a chi-square table. For numeric columns it is mutual information from scikit-learn (`mutual_info_classif`), which is 0 when the column says nothing about the protected attribute and grows as it says more.

**What it shows.** Columns that are tied to the protected attribute. Removing the protected attribute from a model does not remove its influence if such columns stay.

**What it does not show.** The two scales are different (V runs from 0 to 1, mutual information has no fixed top), so compare columns within one kind. And a strong tie is not wrong in itself: age and a medical condition can be related for good reasons. It marks columns to look at.

In [ ]:
protected_values = df[PROTECTED].fillna('missing').astype(str)
association_rows = []
for column in CATEGORICAL:
    table = pd.crosstab(df[column].fillna('missing'), protected_values)
    association_rows.append({'column': column, 'kind': 'categorical', 'Cramers V': association(table.to_numpy(), method='cramer'), 'mutual information': np.nan})
numeric_scores = mutual_info_classif(df[NUMERIC].fillna(df[NUMERIC].median()), protected_values, random_state=SEED)
for column, score in zip(NUMERIC, numeric_scores):
    association_rows.append({'column': column, 'kind': 'numeric', 'Cramers V': np.nan, 'mutual information': score})
associations = pd.DataFrame(association_rows).sort_values(['kind', 'Cramers V', 'mutual information'], ascending=False).reset_index(drop=True)
associations.round(3)

### Step 3: the proxy test, can the other columns predict the protected attribute?

**What this does.** It trains a simple classifier (logistic regression) to predict `PROTECTED` from the columns listed in `NUMERIC` and `CATEGORICAL` in the "Adapt here" cell, which must not include `PROTECTED` itself, and measures how well it does with 5-fold cross-validation. It compares that with a classifier that always guesses the most common value. **Balanced accuracy** averages the share right in each group, so guessing the most common value gets 1 divided by the number of groups: 0.5 for two groups, as in the example, and 0.2 for five. A result well above that guessing line means the columns, together, carry the protected attribute: proxies exist.

**What it shows.** Whether a model that never sees the protected attribute could still reconstruct it. If it can, the model could use it too.

**What it does not show.** Which columns do the work (Step 2 helps), or whether a model that does use it is unfair. And a low score does not clear a dataset: proxies can also work through combinations a simple classifier misses.

In [ ]:
inputs = df[NUMERIC + CATEGORICAL].copy()
inputs[CATEGORICAL] = inputs[CATEGORICAL].fillna('missing').astype(str)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
proxy_model = make_pipeline(
    ColumnTransformer([('numbers', StandardScaler(), NUMERIC), ('categories', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL)]),
    LogisticRegression(max_iter=1000))
predicted = cross_val_predict(proxy_model, inputs, protected_values, cv=folds)
guessing = cross_val_predict(DummyClassifier(strategy='most_frequent'), inputs, protected_values, cv=folds)
proxy = pd.Series({
    'balanced accuracy, guessing the common value': balanced_accuracy_score(protected_values, guessing),
    'balanced accuracy, from the other columns': balanced_accuracy_score(protected_values, predicted),
})
proxy_signal = proxy.iloc[1] - proxy.iloc[0] >= PROXY_MARGIN
print(proxy.round(3).to_string())
print('proxy signal:', 'yes, other columns predict the protected attribute well above guessing' if proxy_signal else 'not above the margin you set')

### Step 4: labelling bias, do annotators agree, and for everyone?

**What this does.** When people label the data, their labels can disagree, and the disagreement can fall unevenly. This step uses the synthetic table of three annotators. It reports **Cohen's kappa** for each pair of annotators (agreement beyond what chance would give, where 1 is perfect and 0 is no better than chance) and **Fleiss' kappa** for all three together (statsmodels), first overall and then inside each group of items.

**What it shows.** Whether labels are reliable, and whether reliability differs by group. Low agreement in one group means the label is applied less consistently there: the items may be harder, the guidelines may not fit them, or one annotator may apply a different standard. A model trained on such labels learns noise or one annotator's habit for that group. The second table, the share of items each annotator labelled 1 in each group, shows which annotator departs from the others and in which direction.

**What it does not show.** Who is right. Agreement is not accuracy: annotators can agree and all be wrong. It also cannot find a bias all annotators share.

In [ ]:
def agreement(frame):
    pairs = {}
    for i, a in enumerate(ANNOTATOR_COLUMNS):
        for b in ANNOTATOR_COLUMNS[i + 1:]:
            pairs[f'{a} vs {b}'] = cohen_kappa_score(frame[a], frame[b])
    table, _ = aggregate_raters(frame[ANNOTATOR_COLUMNS].to_numpy())
    pairs['all three (Fleiss)'] = fleiss_kappa(table, method='fleiss')
    return pairs


agreement_rows = {'all items': agreement(annotations)}
for label, part in annotations.groupby(ANNOTATION_GROUP):
    agreement_rows[f'{label} ({len(part)} items)'] = agreement(part)
agreement_table = pd.DataFrame(agreement_rows).T
share_positive = annotations.groupby(ANNOTATION_GROUP)[ANNOTATOR_COLUMNS].mean()
print('kappa (1 is perfect agreement, 0 is chance):')
print(agreement_table.round(3).to_string())
print()
print('share of items each annotator labelled 1, by group of items:')
print(share_positive.round(3).to_string())

### Step 5: measurement bias, what the label and the columns really measure

**What this does.** No code, only a prompt to read the columns and the label as measurements and ask how each was made.

- **The label** is a record, not the truth. Here `income` is whether a person earned over 50,000 US dollars in 1994. A person who was paid less for the same work, or not paid at all because they did unpaid care work, is labelled low. For your data, ask who or what decided the label, and whether the process was the same for everyone.
- **The groups** are categories somebody chose. Two sexes, five race categories and a country of origin are 1994 US census choices. Groups that matter where you work (language, region, disability, displacement) may not exist as columns, so they cannot be checked at all.
- **A proxy can be a measurement.** Hours per week, for example, measures paid work and misses unpaid work.
- **Missing is a measurement too.** Who is missing from the data, and why (see C-03)?

**What it shows.** Where a number can look objective and still carry a human choice.

**What it does not show.** Anything the data cannot reveal. Only people who know how it was collected can answer.

In [ ]:
# The overview picture, not part of Step 5: the label rate by intersection from Step 1 (groups under MIN_GROUP_SIZE left out) and Fleiss' kappa by group of items from Step 4. It is saved with the evidence.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
overall = label_rates[label_rates.grouping == GROUP_COLUMNS[0]]
first_two = label_rates[(label_rates.grouping == 'intersection') & (~label_rates.small)].sort_values('favourable share')
axes[0].errorbar(first_two['favourable share'], range(len(first_two)),
                 xerr=[first_two['favourable share'] - first_two['low'], first_two['high'] - first_two['favourable share']], fmt='o', capsize=3)
axes[0].set_yticks(range(len(first_two)))
axes[0].set_yticklabels(first_two['group'], fontsize=7)
axes[0].set_xlabel('share with the favourable label, 95% interval')
axes[0].set_title('Label rate by intersection (small groups left out)')
agreement_table.iloc[:, -1].plot.barh(ax=axes[1])
axes[1].set_xlabel("Fleiss' kappa")
axes[1].set_title('Annotator agreement (synthetic items)')
fig.tight_layout()

### Stop and decide: which differences reflect the world, and which reflect how the data was made

The notebook found differences. It cannot tell you where they come from.

Questions to ask in the room, including someone who knows how the data was collected:

1. For each large gap in label rates (Step 1): is it a real difference in the world, a difference in how people were treated, or a difference in how the label was recorded? What evidence would tell them apart?
2. For each column tied to the protected attribute (Steps 2 and 3): is there a good reason to keep it? If you remove it, what is lost, and for whom?
3. If annotators disagree more in one group (Step 4): who should relabel, who decides what the label means, and should a model be trained before that is settled?
4. Which groups matter for your use but are not in the data (Step 5)? How will you find out whether the model serves them?
5. Are the two thresholds in the "Adapt here" cell right for your use: `MIN_GROUP_SIZE`, below which a group is flagged instead of read, and `PROXY_MARGIN`, the distance above guessing at which the proxy test reports a signal? The values given are starting points, not a judgment the notebook makes for you.
6. Who decides what to do about each finding, and where is that written down?

Write your choices in the next cell. Leave it blank if you have not decided, and the summary will say so.

In [ ]:
# YOUR CHOICES. The notebook does not fill this in. Plain words are fine.
GAPS_THAT_REFLECT_THE_WORLD = ''     # gaps you judge to be real differences, and your evidence
GAPS_THAT_REFLECT_HOW_DATA_WAS_MADE = ''   # gaps you judge to come from the data's creation, and what you will do
PROXIES_WE_KEEP_OR_DROP = ''         # columns tied to the protected attribute, and the decision for each
LABEL_PROBLEMS_AND_PLAN = ''         # disagreement you found, who relabels, who decides
GROUPS_NOT_IN_THE_DATA = ''          # groups that matter but cannot be checked here

In [ ]:
import zipfile

EVIDENCE_DIR = Path('evidence')
EVIDENCE_DIR.mkdir(exist_ok=True)

# Small tables of rates and scores. They hold no row of the data.
label_rates.round(4).to_csv(EVIDENCE_DIR / 'C-04_label_rates.csv', index=False)
label_tests.round(4).to_csv(EVIDENCE_DIR / 'C-04_label_tests.csv', index=False)
associations.round(4).to_csv(EVIDENCE_DIR / 'C-04_associations.csv', index=False)
proxy.round(4).to_frame('balanced accuracy').to_csv(EVIDENCE_DIR / 'C-04_proxy_test.csv')
agreement_table.round(4).to_csv(EVIDENCE_DIR / 'C-04_annotator_agreement.csv')
fig.savefig(EVIDENCE_DIR / 'C-04_overview.png', dpi=100, bbox_inches='tight')

versions = '\n'.join(f'- {package} {importlib.metadata.version(package)}' for package in PACKAGES)
summary = f"""# C-04 Bias Detection in Data: evidence summary

Attach the file `C-04_evidence.zip`, which the notebook writes next to the `evidence` folder, to the decision it supports (questions Q-PP-11, Q-PP-12, Q-PP-17, Q-PP-19). It holds this notebook's own summary, tables and pictures, numbers only, never the data. The annotator table is synthetic and describes no real people.

## What was run
- Label: {LABEL}, favourable value {FAVOURABLE}; {len(df):,} rows
- Groups compared: {', '.join(GROUP_COLUMNS)} and their intersection; groups under {MIN_GROUP_SIZE} people are flagged
- Protected attribute for the proxy test: {PROTECTED}

## Findings in numbers
- Association of the label with each group column (Cramers V): {'; '.join(f'{r.grouping} {r["Cramers V"]:.3f}' for _, r in label_tests.iterrows())}
- Proxy test: {proxy.iloc[1]:.3f} balanced accuracy from the other columns against {proxy.iloc[0]:.3f} for guessing; proxy signal {'yes' if proxy_signal else 'no'}
- Annotator agreement, Fleiss kappa: {agreement_table.iloc[0, -1]:.3f} overall; by group {', '.join(f'{i} {v:.3f}' for i, v in agreement_table.iloc[1:, -1].items())} (synthetic items)

## Choices made in the stop-and-decide cell
- Gaps that reflect the world: {GAPS_THAT_REFLECT_THE_WORLD or 'not written yet'}
- Gaps that reflect how the data was made: {GAPS_THAT_REFLECT_HOW_DATA_WAS_MADE or 'not written yet'}
- Proxies kept or dropped: {PROXIES_WE_KEEP_OR_DROP or 'not written yet'}
- Label problems and plan: {LABEL_PROBLEMS_AND_PLAN or 'not written yet'}
- Groups not in the data: {GROUPS_NOT_IN_THE_DATA or 'not written yet'}

## Files beside this summary
C-04_label_rates.csv, C-04_label_tests.csv, C-04_associations.csv, C-04_proxy_test.csv, C-04_annotator_agreement.csv, C-04_overview.png

## Package versions
- Python {platform.python_version()}
{versions}

## Data
SHA-256 of the data file used: {DATA_SHA256_USED}
"""
(EVIDENCE_DIR / 'C-04_summary.md').write_text(summary, encoding='utf-8')
# One ZIP of the files this cell wrote, next to the folder, not inside it. Only the files this cell wrote enter it.
# It is built under a .part name and moved into place only when complete, so an error while it is built leaves no ZIP.
Path('C-04_evidence.zip').unlink(missing_ok=True)
Path('C-04_evidence.zip.part').unlink(missing_ok=True)
with zipfile.ZipFile('C-04_evidence.zip.part', 'x', zipfile.ZIP_DEFLATED) as archive:
    for name in [
        'C-04_annotator_agreement.csv', 'C-04_associations.csv', 'C-04_label_rates.csv',
        'C-04_label_tests.csv', 'C-04_overview.png', 'C-04_proxy_test.csv', 'C-04_summary.md'
    ]:
        path = EVIDENCE_DIR / name
        if not path.is_file() or path.is_symlink():
            raise RuntimeError(f'{name} is missing or is a link, so no ZIP was made. Run this notebook again from the top; if the file is a link, delete it first.')
        archive.write(path, name)
os.replace('C-04_evidence.zip.part', 'C-04_evidence.zip')
print(summary)

## What this does not show, and where to go next

- **It does not decide what is fair.** It shows gaps, ties and disagreements. The decision about each is for people who know the setting, and the "Stop and decide" cell records it.
- **It only sees recorded groups.** Anyone who is not in a column cannot be checked, however much they matter.
- **Proxy tests are a floor.** A classifier that cannot reconstruct a protected attribute has not shown the data is clean.
- **The annotator table is synthetic.** Its numbers show what the method looks like when labelling bias exists. They say nothing about real annotators.
- **Labels are made by people and processes.** A clean-looking label can carry a decision nobody wrote down.

Where to go next: C-03 Data Quality & Representativeness Audit (who is missing), C-05 Data Distribution Analysis (small groups and balance), C-13 Subgroup Fairness Metrics (what the model does once it is trained), and the datasheet template, which records how the data and its labels were made.